# Lab 4 - Spark DataFrame



In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir("/content/drive/MyDrive/Uni/Ki-6-Nam-3/DS200")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
!pip install pyspark

In [5]:
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("BigDataAnalytics-Lab4").getOrCreate()

spark.sparkContext.setLogLevel("WARN")


In [6]:
def clean_columns(df):
    for c in df.columns:
        df = df.withColumnRenamed(c, c.replace("﻿","").strip())
    return df

def read_csv(path):
    df = spark.read.option("header", True).option("sep", ";").option("inferSchema", True).csv(str(path))
    return clean_columns(df)
def parse_ts(column_name, fmt="yyyy-MM-dd HH:mm"):
    return coalesce(
        to_timestamp(col(column_name), fmt),
        col(column_name).cast("timestamp")
    )

cwd = Path.cwd()
if (cwd / "Customer_List.csv").exists():
    base_dir = cwd
elif (cwd / "lab4" / "Customer_List.csv").exists():
    base_dir = cwd / "lab4"
else:
    raise FileNotFoundError("Không tìm thấy thư mục dữ liệu lab4")

print("Base directory:", base_dir)

Base directory: /content/drive/MyDrive/Uni/Ki-6-Nam-3/DS200


In [7]:
customers = read_csv(base_dir / "Customer_List.csv")
orders = read_csv(base_dir / "Orders.csv")
order_items = read_csv(base_dir / "Order_Items.csv")
products = read_csv(base_dir / "Products.csv")
reviews = read_csv(base_dir / "Order_Reviews.csv")

print("Customers:", customers.count())
print("Orders:", orders.count())
print("Order items:", order_items.count())
print("Products:", products.count())
print("Reviews:", reviews.count())

Customers: 102727
Orders: 99441
Order items: 112650
Products: 32951
Reviews: 99270


In [8]:
customers.printSchema()
orders.printSchema()
order_items.printSchema()
products.printSchema()
reviews.printSchema()

root
 |-- Customer_Trx_ID: string (nullable = true)
 |-- Subscriber_ID: string (nullable = true)
 |-- Subscribe_Date: date (nullable = true)
 |-- First_Order_Date: date (nullable = true)
 |-- Customer_Postal_Code: string (nullable = true)
 |-- Customer_City: string (nullable = true)
 |-- Customer_Country: string (nullable = true)
 |-- Customer_Country_Code: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Gender: string (nullable = true)

root
 |-- Order_ID: string (nullable = true)
 |-- Customer_Trx_ID: string (nullable = true)
 |-- Order_Status: string (nullable = true)
 |-- Order_Purchase_Timestamp: timestamp (nullable = true)
 |-- Order_Approved_At: timestamp (nullable = true)
 |-- Order_Delivered_Carrier_Date: timestamp (nullable = true)
 |-- Order_Delivered_Customer_Date: timestamp (nullable = true)
 |-- Order_Estimated_Delivery_Date: timestamp (nullable = true)

root
 |-- Order_ID: string (nullable = true)
 |-- Order_Item_ID: integer (nullable = true)
 |-- Produ

## Câu 2. Thống kê tổng số đơn hàng, số lượng khách hàng và người bán

- Đơn hàng: đếm distinct `Order_ID`
- Khách hàng: đếm distinct `Subscriber_ID`
- Người bán: đếm distinct `Seller_ID`


In [9]:
total_orders = orders.select("Order_ID").distinct().count()
total_customers = customers.select("Subscriber_ID").distinct().count()
total_sellers = order_items.select("Seller_ID").distinct().count()

summary_df = spark.createDataFrame([
    ("Total Orders", total_orders),
    ("Total Customers", total_customers),
    ("Total Sellers", total_sellers)
], ["Metric", "Value"])

summary_df.show(truncate=False)


+---------------+-----+
|Metric         |Value|
+---------------+-----+
|Total Orders   |99441|
|Total Customers|99382|
|Total Sellers  |3095 |
+---------------+-----+



## Câu 3. Phân tích số lượng đơn hàng theo quốc gia


In [13]:
import pyspark.sql.functions as F

orders_by_country = (
    orders.join(customers, on="Customer_Trx_ID", how="inner")
    .groupBy("Customer_Country", "Customer_Country_Code")
    .agg(F.countDistinct("Order_ID").alias("Total_Orders"))
    .orderBy(F.col("Total_Orders").desc(), F.col("Customer_Country").asc())
)

orders_by_country.show(50, truncate=False)

+----------------+---------------------+------------+
|Customer_Country|Customer_Country_Code|Total_Orders|
+----------------+---------------------+------------+
|Germany         |DE                   |41754       |
|France          |FR                   |12848       |
|Netherlands     |NL                   |11629       |
|Belgium         |BE                   |5464        |
|Austria         |AT                   |5043        |
|Switzerland     |CH                   |3640        |
|United Kingdom  |GB                   |3382        |
|Poland          |PL                   |2139        |
|Czechia         |CZ                   |2034        |
|Italy           |IT                   |2025        |
|Spain           |ES                   |1651        |
|Portugal        |PT                   |1336        |
|Sweden          |SE                   |975         |
|Denmark         |DK                   |905         |
|Serbia          |RS                   |746         |
|Norway          |NO        

## Câu 4. Phân tích số lượng đơn hàng theo năm, tháng đặt hàng

Yêu cầu sắp xếp:
- Năm tăng dần
- Tháng giảm dần


In [17]:
order_ts = orders.withColumn(
    "Order_Purchase_Timestamp_TS",
    parse_ts("Order_Purchase_Timestamp")
)

orders_by_year_month = order_ts \
  .withColumn("Order_Year", year("Order_Purchase_Timestamp_TS")) \
  .withColumn("Order_Month", month("Order_Purchase_Timestamp_TS")) \
  .groupBy("Order_Year", "Order_Month") \
  .agg(F.countDistinct("Order_ID").alias("No. Orders")) \
  .orderBy(col("Order_Year").asc(), col("Order_Month").desc())

orders_by_year_month.show(50)

+----------+-----------+----------+
|Order_Year|Order_Month|No. Orders|
+----------+-----------+----------+
|      2022|         12|         1|
|      2022|         10|       324|
|      2022|          9|         4|
|      2023|         12|      5673|
|      2023|         11|      7544|
|      2023|         10|      4631|
|      2023|          9|      4285|
|      2023|          8|      4331|
|      2023|          7|      4026|
|      2023|          6|      3245|
|      2023|          5|      3700|
|      2023|          4|      2404|
|      2023|          3|      2682|
|      2023|          2|      1780|
|      2023|          1|       800|
|      2024|         10|         4|
|      2024|          9|        16|
|      2024|          8|      6512|
|      2024|          7|      6292|
|      2024|          6|      6167|
|      2024|          5|      6873|
|      2024|          4|      6939|
|      2024|          3|      7211|
|      2024|          2|      6728|
|      2024|          1|    

## Câu 5. Thống kê điểm đánh giá trung bình và số lượng theo từng mức

Xử lý dữ liệu:
- Ép `Review_Score` về số nguyên
- Chỉ giữ các giá trị hợp lệ từ 1 đến 5
- Các giá trị NULL hoặc ngoại lệ sẽ chuyển thành NULL


In [27]:
reviews_clean = reviews.withColumn(
      "Review_Score_Str",
      F.trim(F.col("Review_Score").cast("string"))
  ).withColumn(
      "Review_Score_Clean",
      F.when(F.col("Review_Score_Str").rlike("^[1-5]$"),
             F.col("Review_Score_Str").cast("int"))
       .otherwise(F.lit(None).cast("int"))
  )

In [28]:
review_quality = reviews_clean.select(
    F.count("*").alias("Total_Rows"),
    F.count("Review_Score_Clean").alias("Valid_Review_Scores"),
    F.sum(F.when(F.col("Review_Score_Clean").isNull(), 1).otherwise(0)).alias("Invalid_Or_Null")
)

In [29]:
review_quality.show()

+----------+-------------------+---------------+
|Total_Rows|Valid_Review_Scores|Invalid_Or_Null|
+----------+-------------------+---------------+
|     99270|              99223|             47|
+----------+-------------------+---------------+



In [30]:
avg_review_score = reviews_clean.select(
    F.round(F.avg("Review_Score_Clean"), 2).alias("Avergage_Review_Score")
)

In [31]:
avg_review_score.show()

+---------------------+
|Avergage_Review_Score|
+---------------------+
|                 4.09|
+---------------------+



In [32]:
score_distribution = (
    reviews_clean
    .groupBy("Review_Score_Clean")
    .agg(F.count("*").alias("Total_Reviews"))
    .orderBy("Review_Score_Clean")
)

In [33]:
score_distribution.show()

+------------------+-------------+
|Review_Score_Clean|Total_Reviews|
+------------------+-------------+
|              NULL|           47|
|                 1|        11424|
|                 2|         3151|
|                 3|         8179|
|                 4|        19141|
|                 5|        57328|
+------------------+-------------+



In [35]:
reviews_clean.filter(F.col("Review_Score_Clean").isNull()) \
    .select("Review_ID", "Order_ID", "Review_Score") \
    .show(20, truncate=False)

+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------+----------------+
|Review_ID                                                                                                                                                                                                                |Order_ID        |Review_Score    |
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------+----------------+
|I do not recommend this partner, I bought a product, they sent me another inferior one, I returned it, which was delivered on 28/02 and to date on 05/03 they have not given me a response. Rating 0.                    |NULL            |NU

# Phần tự chọn
Em chọn làm các câu **6, 8, 10**.


## Câu 6. Tính doanh thu năm 2024 theo danh mục sản phẩm

Doanh thu = `Price + Freight_Value`


In [40]:
orders_2024 = order_ts.filter(year("Order_Purchase_Timestamp_TS") == 2024)

revenue_2024_by_category = orders_2024     .join(order_items, on="Order_ID", how="inner") \
    .join(products, on="Product_ID", how="left") \
    .withColumn("Revenue", col("Price") + col("Freight_Value")) \
    .groupBy("Product_Category_Name").agg(
        F.round(F.sum("Revenue"), 2).alias("Total_Revenue"),
        F.countDistinct("Order_ID").alias("Total_Orders")
    ).orderBy(col("Total_Revenue").desc())

revenue_2024_by_category.show(50, truncate=False)


+---------------------------------------+-------------+------------+
|Product_Category_Name                  |Total_Revenue|Total_Orders|
+---------------------------------------+-------------+------------+
|Health_Beauty                          |885191.12    |5402        |
|Watches_Gifts                          |771986.75    |3493        |
|Bed_Bath_Table                         |650794.7     |4909        |
|Sports_Leisure                         |621999.34    |4058        |
|Computers_Accessories                  |594771.04    |4053        |
|Housewares                             |491576.96    |3421        |
|Furniture_Decor                        |476466.13    |3199        |
|Auto                                   |404210.57    |2457        |
|Baby                                   |299052.56    |1667        |
|Cool_Stuff                             |273910.05    |1415        |
|Garden_Tools                           |259068.32    |1535        |
|Telephony                        

## Câu 8. Hiệu số giữa ngày giao thực tế và ngày giao dự kiến

Sử dụng:
- Thực tế: `Order_Delivered_Carrier_Date`
- Dự kiến: `Shipping_Limit_Date`

`Delay_Days > 0`: giao trễ  
`Delay_Days = 0`: đúng hạn  
`Delay_Days < 0`: giao sớm


In [41]:
delivery_perf = orders.join(order_items, on="Order_ID", how="inner")  \
  .withColumn("Delivered_Carrier_TS", parse_ts("Order_Delivered_Carrier_Date")) \
  .withColumn("Shipping_Limit_TS", parse_ts("Shipping_Limit_Date"))  \
  .withColumn("Delay_Days", datediff(col("Delivered_Carrier_TS"), col("Shipping_Limit_TS")))

# Xem một số bản ghi mẫu

delivery_perf.select(
    "Order_ID", "Product_ID", "Seller_ID",
    "Delivered_Carrier_TS", "Shipping_Limit_TS", "Delay_Days"
).show(20, truncate=False)

+--------------------------------+--------------------------------+--------------------------------+--------------------+-------------------+----------+
|Order_ID                        |Product_ID                      |Seller_ID                       |Delivered_Carrier_TS|Shipping_Limit_TS  |Delay_Days|
+--------------------------------+--------------------------------+--------------------------------+--------------------+-------------------+----------+
|00010242fe8c5a6d1ba2dd792cb16214|4244733e06e7ecb4970a6e2683c13e61|48436dade18ac8b2bce089ec2a041202|2023-09-19 18:34:00 |2023-09-19 09:45:00|0         |
|00018f77f2f0320c557190d7a144bdd3|e5f2d52b802189ee658865ca93d83a8f|dd7ddc04e1b6c2c614352b383efe2d36|2023-05-04 14:35:00 |2023-05-03 11:05:00|1         |
|000229ec398224ef6ca0657da4fc703e|c777355d18b72b67abbeef9df44fd0fd|5b51032eddd242adc84c38acab88f23d|2024-01-16 12:36:00 |2024-01-18 14:48:00|-2        |
|00024acbcdf0a6daa1e931b038114c75|7634da152a4610f1595efa32f14722fc|9d7a1d34a505240

In [42]:
delivery_summary = delivery_perf.select(
    count("*").alias("Total_Rows"),
    count("Delay_Days").alias("Rows_With_Delay"),
    round(avg("Delay_Days"), 2).alias("Avg_Delay_Days"),
    min("Delay_Days").alias("Min_Delay_Days"),
    max("Delay_Days").alias("Max_Delay_Days")
)

status_distribution = delivery_perf.withColumn(
    "Delivery_Status",
    when(col("Delay_Days") > 0, "Late")
    .when(col("Delay_Days") == 0, "On Time")
    .when(col("Delay_Days") < 0, "Early")
    .otherwise("Unknown")
).groupBy("Delivery_Status").agg(count("*").alias("Total")).orderBy(col("Total").desc())

delivery_summary.show()
status_distribution.show()


+----------+---------------+--------------+--------------+--------------+
|Total_Rows|Rows_With_Delay|Avg_Delay_Days|Min_Delay_Days|Max_Delay_Days|
+----------+---------------+--------------+--------------+--------------+
|    112650|         111456|         -3.44|         -1055|           118|
+----------+---------------+--------------+--------------+--------------+

+---------------+-----+
|Delivery_Status|Total|
+---------------+-----+
|          Early|97187|
|        On Time| 7599|
|           Late| 6670|
|        Unknown| 1194|
+---------------+-----+



## Câu 10. Xếp hạng seller dựa trên tổng doanh thu và số lượng đơn hàng


In [43]:
seller_ranking = order_items \
    .withColumn("Revenue", col("Price") + col("Freight_Value")) \
    .groupBy("Seller_ID") \
    .agg(
        F.round(F.sum("Revenue"), 2).alias("Total_Revenue"),
        F.countDistinct("Order_ID").alias("Total_Orders")
    )

ranking_window = Window.orderBy(col("Total_Revenue").desc(), col("Total_Orders").desc())

seller_ranking = seller_ranking \
    .withColumn("Rank", dense_rank().over(ranking_window)) \
    .select("Rank", "Seller_ID", "Total_Revenue", "Total_Orders") \
    .orderBy("Rank")

seller_ranking.show(50, truncate=False)


+----+--------------------------------+-------------+------------+
|Rank|Seller_ID                       |Total_Revenue|Total_Orders|
+----+--------------------------------+-------------+------------+
|1   |4869f7a5dfa277a7dca6462dcf3b52b2|249640.7     |1132        |
|2   |7c67e1448b00f6e969d365cea6b010ab|239536.44    |982         |
|3   |53243585a1d6dc2643021fd1853d8905|235856.68    |358         |
|4   |4a3ca9315b744ce9f8e9374361493884|235539.96    |1806        |
|5   |fa1c13f2614d7b5c4749cbc52fecda94|204084.73    |585         |
|6   |da8622b14eb17ae2831f4ac5b9dab84a|185192.32    |1314        |
|7   |7e93a43ef30c4f03f38b393420bc753a|182754.05    |336         |
|8   |1025f0e2d44d7041d6cf58b6550e0bfa|172860.69    |915         |
|9   |7a67c85e85bb2ce8582c35f2203ad736|162648.38    |1160        |
|10  |955fee9216a65b617aa5c0531780ce60|160602.68    |1287        |
|11  |6560211a19b47992c3666cc44a7e94c0|151265.77    |1854        |
|12  |1f50f920176fa81dab994f9023523100|142104.98    |1404     

In [44]:
spark.stop()
